# 11.16 - Agentic RAG

**Phase:** 11 - RAG Systems

**Status:** VERIFIED

---

## 1. What Are We Solving?

Not every query needs retrieval. Agentic RAG lets the LLM decide whether to retrieve, which tool to use, and whether it needs more information - adapting the pipeline to the query instead of always running the same steps.

## 2. Why Does This Matter?

Dynamic routing saves cost and latency on queries that need no lookup, and enables multi-hop retrieval for complex questions. This bridges RAG and agents (Phase 14).

## 3. Prerequisites

Unit 11.15 (Advanced RAG), Phase 10 (LLMs).

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Build a manual tool-use loop (classify -> choose tool -> run -> evaluate)
- Use llm() for decisions with deterministic fallbacks
- Add a max 3-step loop guard; no external network calls

## 5. Mental Model

Agentic RAG is a researcher who decides: 'Do I need to look this up? Where? Do I need more?' They don't always visit the library.

```text
Query -> decide retrieve? -> tool (kb/web/refuse) -> evaluate -> maybe again -> answer
```


## 6. Setup
LLM helper + a tiny knowledge base (default retrieval tool), all offline-safe.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from typing import Annotated
import operator

GROQ_MODEL = os.environ.get("GROQ_MODEL", "openai/gpt-oss-20b")


def llm(prompt: str, model: str = GROQ_MODEL, temperature: float = 0.0) -> str:
    """One-shot Groq call with a deterministic mock fallback."""
    if not os.environ.get("GROQ_API_KEY"):
        return "mock: deterministic model output for offline runs."
    try:
        from langchain_groq import ChatGroq
        chat = ChatGroq(model=model, temperature=temperature)
        return chat.invoke(prompt).content.strip()
    except Exception as e:
        return f"[llm-error: {type(e).__name__}]"


print("Groq model:", GROQ_MODEL)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))


Groq model: openai/gpt-oss-20b
GROQ_API_KEY present: True


In [2]:
KB = {
    "return": "Returns are accepted within 30 days in original packaging.",
    "shipping": "Standard shipping takes 5-7 business days.",
    "refund": "Refunds post within 5-7 business days of receipt.",
}


## 7. Tool Definitions
Three deterministic tools: `kb_search` (exact keyword lookup in KB), `web_style_search` (simulated external search - no network), and `refuse` (no answer). Each returns a simple dict.

In [3]:
def kb_search(query):
    q = query.lower()
    for key, text in KB.items():
        if key in q:
            return {"tool": "kb_search", "ok": True, "text": text}
    return {"tool": "kb_search", "ok": False, "text": ""}


def web_style_search(query):
    # deterministic stand-in for an external / live search (never hits the network)
    q = query.lower()
    if "price" in q or "compare" in q:
        return {"tool": "web_style", "ok": True,
                "text": "mock web result: pricing varies, see vendor pages."}
    return {"tool": "web_style", "ok": False, "text": ""}


def refuse(query):
    return {"tool": "refuse", "ok": False, "text": "I cannot answer this question."}

TOOLS = {"kb_search": kb_search, "web_style_search": web_style_search, "refuse": refuse}


## 8. The Agent Loop
The agent uses `llm()` to (1) choose a tool, then (2) decide if the result is sufficient. Deterministic fallbacks kick in when the LLM is a mock or errors. A `max_steps=3` guard prevents infinite loops.

In [4]:
def classify_tool(query, fallback="kb_search"):
    out = llm(f"Which tool? kb_search, web_style_search, or refuse. Query: {query}")
    for tool in TOOLS:
        if tool in out:
            return tool
    # deterministic fallback (also used when llm returns the mock string)
    if any(w in query.lower() for w in ("price", "compare", "latest")):
        return "web_style_search"
    if any(w in query.lower() for w in ("return", "shipping", "refund")):
        return "kb_search"
    return "refuse"


def run_agent(query, max_steps=3):
    trace, result = [], {}
    for step in range(1, max_steps + 1):
        tool = classify_tool(query)
        trace.append(f"step{step}: classify -> {tool}")
        res = TOOLS[tool](query)
        trace.append(f"step{step}: {tool} ok={res['ok']}")
        if res["ok"]:
            result = res
            break
        if tool == "refuse":
            result = res
            break
        trace.append(f"step{step}: result insufficient -> retry")
    return trace, result


queries = ["How do I return a product?", "What is the price of the premium plan?", "Thanks"]
for q in queries:
    trace, result = run_agent(q)
    print("QUERY:", q)
    for t in trace:
        print("   ", t)
    print("   FINAL:", result.get("text", "(none)") if result else "(failed after guard)")
    print()


QUERY: How do I return a product?
    step1: classify -> kb_search
    step1: kb_search ok=True
   FINAL: Returns are accepted within 30 days in original packaging.



QUERY: What is the price of the premium plan?
    step1: classify -> web_style_search
    step1: web_style_search ok=True
   FINAL: mock web result: pricing varies, see vendor pages.



QUERY: Thanks
    step1: classify -> refuse
    step1: refuse ok=False
   FINAL: I cannot answer this question.



## 9. Sufficiency Check (optional second hop)
For multi-step behaviour, the agent can decide after a result whether it needs another retrieval. We wire a deterministic gate: if `kb_search` found nothing and a web search is possible, go again - never more than the step guard.

In [5]:
def multi_step(query):
    trace, result, guard = [], {}, 0
    tool = "kb_search"
    while guard < 3:
        guard += 1
        res = TOOLS[tool](query)
        trace.append(f"hop{guard}: {tool} ok={res['ok']}")
        if res["ok"]:
            result = res
            break
        if tool == "kb_search" and not res["ok"]:
            tool = "web_style_search"
            trace.append("hop: escalating kb -> web")
            continue
        result = refuse(query)
        break
    return trace, result


q = "What is the price of the premium plan?"
tr, res = multi_step(q)
for t in tr:
    print("  ", t)
print("   FINAL:", res.get("text"))


   hop1: kb_search ok=False
   hop: escalating kb -> web
   hop2: web_style_search ok=True
   FINAL: mock web result: pricing varies, see vendor pages.


## 10. Trace + Reflection
Print the decision trace so you can see *why* the agent chose each path. Logging decisions is the #1 debugging tool for agentic systems.

In [6]:
for q in ["How do I return a product?",
          "What is the price of the premium plan?",
          "How does our return policy compare to competitors?"]:
    tr, result = run_agent(q)
    print(f"Q: {q}")
    print("   trace ->", " ; ".join(tr))
    print("   final ->", result.get("text", "(none)") if result else "(aborted)")
    print()


Q: How do I return a product?
   trace -> step1: classify -> kb_search ; step1: kb_search ok=True
   final -> Returns are accepted within 30 days in original packaging.



Q: What is the price of the premium plan?
   trace -> step1: classify -> web_style_search ; step1: web_style_search ok=True
   final -> mock web result: pricing varies, see vendor pages.



Q: How does our return policy compare to competitors?
   trace -> step1: classify -> web_style_search ; step1: web_style_search ok=True
   final -> mock web result: pricing varies, see vendor pages.




## Common Mistakes

- Over-engineering when naive RAG works fine.
- Agent adds latency for every query (incl. ones needing no retrieval).
- Not evaluating whether agent decisions improve quality.
- Agent loops indefinitely without a guard.

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| Agent never retrieves | Decision threshold too high | Adjust decision prompt |
| Agent always retrieves | Prompt too aggressive | Add NO_RETRIEVE examples |
| Agent loops | No exit on insufficiency | Add step limit |
| High latency | Too many LLM calls | Cache decisions, reduce steps |

## Best Practices

- Start with fixed retrieval; add agentic behaviour only when needed.
- Log agent decisions for debugging.
- Set hard limits on retrieval iterations.
- Evaluate whether agent decisions actually improve quality.
- Consider the latency cost of each decision.

## Hands-On Practice

1. **Basic:** Implement a retrieve-or-not decision for 10 queries.
2. **Guided:** Add a sufficiency check after retrieval.
3. **Independent:** Implement multi-step retrieval.
4. **Realistic:** Compare fixed vs agentic RAG on 30 queries.
5. **Challenge:** Build a self-RAG system with reflection.

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.
